In [8]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
df_old=pd.read_csv('full_training_data.csv')

In [3]:
df_old.head(2)

,player_id,name,role,round,home_average,away_average,overall_average,current_price,matches_played,rating,...,location_adjusted_average,matchup_strength,team_expected_performance,delantero_matchup_bonus,centrocampista_matchup_bonus,defensa_matchup_bonus,portero_matchup_bonus,home_matchup_boost,difficult_matchup,easy_matchup
0,51ffb2540ac2ec8b0700001e,Dani Rodríguez,centrocampista,13,2.0,0.0,2.000000,1000000,2,1,...,0.0,-0.528376,0.605592,0.0,0.160761,0.0,0.0,0.000000,1,0
1,52013ee178b20d7f07000351,Josan,centrocampista,13,4.4,1.5,3.571429,1000000,7,1,...,4.4,-0.570438,0.555189,0.0,0.150752,0.0,0.0,0.062813,1,0


In [5]:
df_lamine=df_old[df_old['name']=='Lamine Yamal']

In [6]:
df_lamine[['round', 'match_minus_1', 'match_minus_2', 'target_points']]

,round,match_minus_1,match_minus_2,target_points
329,13,9,10,11.0
845,14,11,9,7.0
1362,15,7,11,9.0
1877,16,9,7,NaN


In [7]:
username = 'rodrigo'
host = 'localhost'           
port = '5432'               
database = 'futmondo_full_players_info'

In [9]:
engine = create_engine(f'postgresql+psycopg2://{username}@{host}:{port}/{database}')

In [57]:
query = "SELECT * FROM full_training_data"
df = pd.read_sql(query, engine)
df.shape

(4677, 39)

In [59]:
df.shape

(4677, 39)

In [21]:
import pandas as pd

# 1. Filter the old dataframe for rounds 13, 14, 15
old_filtered = df_old[df_old['round'].isin([13, 14, 15, 16])]

# 2. Filter the new dataframe for rounds 16 through 21
new_filtered = df[df['round'].isin([17, 18, 19, 20, 21])]

# 3. Concatenate them vertically
df_combined = pd.concat([old_filtered, new_filtered], axis=0).reset_index(drop=True)

In [31]:
import pandas as pd
import numpy as np

# 1. Calculate the mean target_points per player (excluding round 16 AND round 21)
# We exclude 21 as well to ensure the mean is based only on "historical" known data
mean_points = (
    df_combined[~df_combined['round'].isin([16, 21])]
    .groupby('player_id')['target_points']
    .mean()
)

# 2. Update 'target_points' ONLY for round 16
mask_16 = df_combined['round'] == 16
df_combined.loc[mask_16, 'target_points'] = df_combined.loc[mask_16, 'player_id'].map(mean_points)

# 3. Update 'match_minus_1' ONLY for round 17
mask_17 = df_combined['round'] == 17
df_combined.loc[mask_17, 'match_minus_1'] = df_combined.loc[mask_17, 'player_id'].map(mean_points)

# 4. Explicitly ensure round 21 target_points are NaN
df_combined.loc[df_combined['round'] == 21, 'target_points'] = np.nan

In [32]:
df_gonzalo=df_combined[df_combined['name']=='Gonzalo']

In [33]:
df_gonzalo[['round', 'match_minus_1', 'match_minus_2', 'target_points']]

,round,match_minus_1,match_minus_2,target_points
438,13,2.000000,0,0.000000
954,14,0.000000,2,0.000000
1471,15,0.000000,0,2.000000
1986,16,2.000000,0,3.571429
2550,17,3.571429,0,1.000000
3036,18,1.000000,2,0.000000
3555,19,0.000000,1,20.000000
4069,20,20.000000,0,2.000000
4583,21,2.000000,20,NaN


In [34]:
df_combined.shape

(4677, 39)

In [35]:
df_combined['round'].value_counts().sort_index()

round
13    516
14    516
15    516
16    516
17    533
18    522
19    522
20    519
21    517
Name: count, dtype: int64

In [36]:
db_config = {
    'username': 'rodrigo',
    'host': 'localhost',
    'port': '5432',
    'database': 'futmondo_full_players_info'}

engine = create_engine(
    f"postgresql+psycopg2://{db_config['username']}@{db_config['host']}:"
    f"{db_config['port']}/{db_config['database']}")

# # Load existing player points data
# query = "SELECT * FROM player_points"
# df = pd.read_sql(query, engine)
# df.shape

In [37]:
df_combined.to_sql('full_training_data', engine, if_exists='replace', index=False)

487

In [3]:
from sqlalchemy import create_engine
from src.utils import (
    create_round_features,
    add_matchup_probabilities,
    create_advanced_features,
    get_team_stats,
    standardize_team_names,
    predict_upcoming_matches)
import pandas as pd

In [5]:
# Connection to database
db_config = {
    'username': 'rodrigo',
    'host': 'localhost',
    'port': '5432',
    'database': 'futmondo_full_players_info'}

engine = create_engine(
    f"postgresql+psycopg2://{db_config['username']}@{db_config['host']}:"
    f"{db_config['port']}/{db_config['database']}")

In [6]:
# 3. Load matches data and predict upcoming matches
df_la_liga = pd.read_sql("SELECT * FROM la_liga_matches", engine) # Data with probabilities
df_liga_next = pd.read_csv('/Users/rodrigo/football-data-analytics/futmondo_points_predict/data/la_liga_next_rounds copy.csv')

# Clean and standardize team names
df_la_liga = standardize_team_names(df_la_liga, is_historical=True)
df_liga_next = standardize_team_names(df_liga_next, is_historical=False)

historical_teams = set(df_la_liga['HomeTeam'].unique()) | set(df_la_liga['AwayTeam'].unique())
upcoming_teams = set(df_liga_next['Home Team'].unique()) | set(df_liga_next['Away Team'].unique())
missing_teams = upcoming_teams - historical_teams

team_stats = get_team_stats(df_la_liga)
predictions_df = predict_upcoming_matches(df_liga_next, df_la_liga, team_stats)
updated_df = pd.concat([df_la_liga, predictions_df], ignore_index=True)  # combine historical and predicted matches


/var/folders/r5/bx2jhb6n64n4zm91gw712nr00000gn/T/ipykernel_69623/1756025752.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated_df = pd.concat([df_la_liga, predictions_df], ignore_index=True)  # combine historical and predicted matches


In [8]:
updated_df.to_sql('la_liga_matches', engine, if_exists='replace', index=False)

350